In [1]:
import pandas as pd
import numpy as np
pd.options.display.max_columns = None
from openslide import open_slide
import glob

In [2]:
def filter_files(x, slides):
    if slides == 'ffpe':
        if '-DX' in x or 'rna_seq' in x:
            return True
        else:
            return False
    elif slides == 'ff':
        if '-TS' in x or 'rna_seq' in x:
            return True
        else:
            return False
            
    return None


def has_magnif(image_path):
    img = open_slide(image_path)
    magnification = img.properties.get("aperio.AppMag", None)
    if magnification in ["40", "20"]:
        return True
    else:
        return False

def get_slide_orientation(x):
    if '-TS' in x:
        return 'TS'
    elif '-BS' in x:
        return 'BS'
    elif 'rna_seq' in x:
        return 'RNA'
    elif '-DX' in x:
        return 'DX'
    else:
        return np.nan 

In [3]:
slide_type = 'ffpe' # ff
sample_sheet = 'gdc_sample_sheet.2025-04-11'

In [4]:
df = pd.read_csv(f'data/{sample_sheet}.tsv', sep='\t')
df = df[df['File Name'].apply(lambda x: filter_files(x, slides=slide_type))]
df

,File ID,File Name,Data Category,Data Type,Project ID,Case ID,Sample ID,Tissue Type,Tumor Descriptor,Specimen Type,Preservation Method
3,1123d866-9b24-47e1-a285-20a374d62df0,TCGA-60-2712-01Z-00-DX1.97003dfc-4b37-4491-86e...,Biospecimen,Slide Image,TCGA-LUSC,TCGA-60-2712,TCGA-60-2712-01Z,Tumor,Primary,Solid Tissue,FFPE
5,044d098e-c65f-4152-8939-3c315aed0073,80ca4c12-21e8-45d1-8820-537b99bce32d.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-LUSC,TCGA-60-2712,TCGA-60-2712-01A,Tumor,Primary,Unknown,Unknown
7,f95d3d81-2433-4018-a042-f24b9f04946a,545f9937-b128-4f44-8b12-ded0fb79bf3f.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-LUSC,TCGA-56-7221,TCGA-56-7221-01A,Tumor,Primary,Solid Tissue,Unknown
9,c863b3b5-ebd2-43b6-99d6-d5741e872ff3,TCGA-56-7221-01Z-00-DX1.f897f1ee-2796-4183-931...,Biospecimen,Slide Image,TCGA-LUSC,TCGA-56-7221,TCGA-56-7221-01Z,Tumor,Primary,Solid Tissue,FFPE
11,420eca55-8832-46ec-bd6f-28add7b8b6b7,TCGA-21-A5DI-01Z-00-DX1.E9123261-ADE7-468C-9E9...,Biospecimen,Slide Image,TCGA-LUSC,TCGA-21-A5DI,TCGA-21-A5DI-01Z,Tumor,Primary,Solid Tissue,FFPE
...,...,...,...,...,...,...,...,...,...,...,...
2742,53adb84c-8f77-4786-94f4-49000e4bb2fe,TCGA-66-2783-01Z-00-DX1.34df2ea9-f8d8-448e-bd5...,Biospecimen,Slide Image,TCGA-LUSC,TCGA-66-2783,TCGA-66-2783-01Z,Tumor,Primary,Solid Tissue,FFPE
2745,c8a68cbe-87a1-44b3-bf4b-68ba0de91994,3c174eac-9153-4c66-8da4-969925c2c4c4.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-LUSC,TCGA-66-2783,TCGA-66-2783-01A,Tumor,Primary,Unknown,Unknown
2747,5c3ec03d-d9eb-48e8-b132-5733dcf83131,5332a617-f178-4197-8ed4-2f49bf8751e0.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-LUSC,TCGA-66-2795,TCGA-66-2795-01A,Tumor,Primary,Solid Tissue,Unknown
2749,42afded2-aa23-4f7f-b86f-ac47b324db1d,fbf226a9-b07a-4aa2-bcb6-ceaac4b373c0.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-LUSC,TCGA-66-2788,TCGA-66-2788-01A,Tumor,Primary,Solid Tissue,Unknown


In [5]:
if slide_type == 'ffpe':
    paired_samples = df[['Case ID', 'Data Type']].value_counts().reset_index().value_counts('Case ID').reset_index()
elif slide_type == 'ff':
    paired_samples = df[['Sample ID', 'Data Type']].value_counts().reset_index().value_counts('Sample ID').reset_index()
paired_samples = paired_samples[paired_samples['count'] > 1]
paired_samples

,Case ID,count
0,TCGA-XC-AA0X,2
1,TCGA-18-3406,2
2,TCGA-18-3407,2
3,TCGA-18-3408,2
4,TCGA-18-3409,2
...,...,...
471,TCGA-21-1080,2
472,TCGA-21-1081,2
473,TCGA-21-1082,2
474,TCGA-21-1083,2


In [6]:
if slide_type == 'ffpe':
    df = df[df['Case ID'].isin(paired_samples['Case ID'].values)]
elif slide_type == 'ff':
    df = df[df['Sample ID'].isin(paired_samples['Sample ID'].values)]
df

,File ID,File Name,Data Category,Data Type,Project ID,Case ID,Sample ID,Tissue Type,Tumor Descriptor,Specimen Type,Preservation Method
3,1123d866-9b24-47e1-a285-20a374d62df0,TCGA-60-2712-01Z-00-DX1.97003dfc-4b37-4491-86e...,Biospecimen,Slide Image,TCGA-LUSC,TCGA-60-2712,TCGA-60-2712-01Z,Tumor,Primary,Solid Tissue,FFPE
5,044d098e-c65f-4152-8939-3c315aed0073,80ca4c12-21e8-45d1-8820-537b99bce32d.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-LUSC,TCGA-60-2712,TCGA-60-2712-01A,Tumor,Primary,Unknown,Unknown
7,f95d3d81-2433-4018-a042-f24b9f04946a,545f9937-b128-4f44-8b12-ded0fb79bf3f.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-LUSC,TCGA-56-7221,TCGA-56-7221-01A,Tumor,Primary,Solid Tissue,Unknown
9,c863b3b5-ebd2-43b6-99d6-d5741e872ff3,TCGA-56-7221-01Z-00-DX1.f897f1ee-2796-4183-931...,Biospecimen,Slide Image,TCGA-LUSC,TCGA-56-7221,TCGA-56-7221-01Z,Tumor,Primary,Solid Tissue,FFPE
11,420eca55-8832-46ec-bd6f-28add7b8b6b7,TCGA-21-A5DI-01Z-00-DX1.E9123261-ADE7-468C-9E9...,Biospecimen,Slide Image,TCGA-LUSC,TCGA-21-A5DI,TCGA-21-A5DI-01Z,Tumor,Primary,Solid Tissue,FFPE
...,...,...,...,...,...,...,...,...,...,...,...
2742,53adb84c-8f77-4786-94f4-49000e4bb2fe,TCGA-66-2783-01Z-00-DX1.34df2ea9-f8d8-448e-bd5...,Biospecimen,Slide Image,TCGA-LUSC,TCGA-66-2783,TCGA-66-2783-01Z,Tumor,Primary,Solid Tissue,FFPE
2745,c8a68cbe-87a1-44b3-bf4b-68ba0de91994,3c174eac-9153-4c66-8da4-969925c2c4c4.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-LUSC,TCGA-66-2783,TCGA-66-2783-01A,Tumor,Primary,Unknown,Unknown
2747,5c3ec03d-d9eb-48e8-b132-5733dcf83131,5332a617-f178-4197-8ed4-2f49bf8751e0.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-LUSC,TCGA-66-2795,TCGA-66-2795-01A,Tumor,Primary,Solid Tissue,Unknown
2749,42afded2-aa23-4f7f-b86f-ac47b324db1d,fbf226a9-b07a-4aa2-bcb6-ceaac4b373c0.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-LUSC,TCGA-66-2788,TCGA-66-2788-01A,Tumor,Primary,Solid Tissue,Unknown


In [7]:
metadata = []
if slide_type == 'ffpe':
    iterate_over = 'Case ID'
elif slide_type == 'ff':
    iterate_over = 'Sample ID'

for iterate_over_id in df[iterate_over].unique():
    tab = df[df[iterate_over] == iterate_over_id]

    tab_rna = tab[tab['Data Type'] == 'Gene Expression Quantification']
    tab_slide = tab[tab['Data Type'] == 'Slide Image']
    

    for _, row_rna in tab_rna.iterrows():
        for _, row_slide in tab_slide.iterrows():
            if row_rna['Tumor Descriptor'] == row_slide['Tumor Descriptor']:
                metadata.append([row_slide['File Name'], row_rna['File Name'], row_rna['Case ID'], 
                                 row_slide['Sample ID'], row_rna['Sample ID'], row_slide['Tumor Descriptor']])
                
metadata = pd.DataFrame(metadata, columns=['image_path', 'rna_path', 'case_id', 'sample_slide_id', 'sample_rna_id', 'sample_type'])
metadata

,image_path,rna_path,case_id,sample_slide_id,sample_rna_id,sample_type
0,TCGA-60-2712-01Z-00-DX1.97003dfc-4b37-4491-86e...,80ca4c12-21e8-45d1-8820-537b99bce32d.rna_seq.a...,TCGA-60-2712,TCGA-60-2712-01Z,TCGA-60-2712-01A,Primary
1,TCGA-56-7221-01Z-00-DX1.f897f1ee-2796-4183-931...,545f9937-b128-4f44-8b12-ded0fb79bf3f.rna_seq.a...,TCGA-56-7221,TCGA-56-7221-01Z,TCGA-56-7221-01A,Primary
2,TCGA-21-A5DI-01Z-00-DX1.E9123261-ADE7-468C-9E9...,9b86812f-b1ee-4b6d-9691-8587f2487c4a.rna_seq.a...,TCGA-21-A5DI,TCGA-21-A5DI-01Z,TCGA-21-A5DI-01A,Primary
3,TCGA-43-7657-01Z-00-DX1.d8a5d257-c5ca-4192-b6a...,6df80c92-775e-4bcf-b2c7-6cbd7e147447.rna_seq.a...,TCGA-43-7657,TCGA-43-7657-01Z,TCGA-43-7657-01A,Primary
4,TCGA-94-7033-01Z-00-DX1.43146ed9-30a5-420d-bd9...,23f1ad0c-c9d5-408f-bba8-1bb71364007b.rna_seq.a...,TCGA-94-7033,TCGA-94-7033-01Z,TCGA-94-7033-01A,Primary
...,...,...,...,...,...,...
515,TCGA-66-2785-01Z-00-DX1.b9439ee1-d22b-4ccd-b53...,b5356a8b-9401-442f-a5dd-7e9273723a67.rna_seq.a...,TCGA-66-2785,TCGA-66-2785-01Z,TCGA-66-2785-01A,Primary
516,TCGA-77-6843-01Z-00-DX1.5ced4995-81a1-4dfd-82b...,0b646082-9e64-4fb2-a9c9-e283d4c49852.rna_seq.a...,TCGA-77-6843,TCGA-77-6843-01Z,TCGA-77-6843-01A,Primary
517,TCGA-66-2781-01Z-00-DX1.ed9ff5b7-7c66-4bf7-bfb...,59fd6a72-d96d-42af-8013-11f43af6eed5.rna_seq.a...,TCGA-66-2781,TCGA-66-2781-01Z,TCGA-66-2781-01A,Primary
518,TCGA-39-5028-01Z-00-DX1.7994ec22-746d-4c30-813...,77b0bb2f-6e9f-4c67-b64c-98eebf0bd99d.rna_seq.a...,TCGA-39-5028,TCGA-39-5028-01Z,TCGA-39-5028-01A,Primary


In [8]:
metadata['data_type_info'] = metadata['image_path'].apply(lambda x: get_slide_orientation(x))
metadata['data_type_info'].value_counts()

data_type_info
DX    520
Name: count, dtype: int64

In [9]:
metadata = metadata[metadata.apply(lambda x: len(glob.glob(f"data/*/*/{x.rna_path}")) > 0, axis=1)]
metadata = metadata[metadata.apply(lambda x: len(glob.glob(f"data/*/*/{x.image_path}")) > 0, axis=1)]
metadata

,image_path,rna_path,case_id,sample_slide_id,sample_rna_id,sample_type,data_type_info
0,TCGA-60-2712-01Z-00-DX1.97003dfc-4b37-4491-86e...,80ca4c12-21e8-45d1-8820-537b99bce32d.rna_seq.a...,TCGA-60-2712,TCGA-60-2712-01Z,TCGA-60-2712-01A,Primary,DX
1,TCGA-56-7221-01Z-00-DX1.f897f1ee-2796-4183-931...,545f9937-b128-4f44-8b12-ded0fb79bf3f.rna_seq.a...,TCGA-56-7221,TCGA-56-7221-01Z,TCGA-56-7221-01A,Primary,DX
2,TCGA-21-A5DI-01Z-00-DX1.E9123261-ADE7-468C-9E9...,9b86812f-b1ee-4b6d-9691-8587f2487c4a.rna_seq.a...,TCGA-21-A5DI,TCGA-21-A5DI-01Z,TCGA-21-A5DI-01A,Primary,DX
3,TCGA-43-7657-01Z-00-DX1.d8a5d257-c5ca-4192-b6a...,6df80c92-775e-4bcf-b2c7-6cbd7e147447.rna_seq.a...,TCGA-43-7657,TCGA-43-7657-01Z,TCGA-43-7657-01A,Primary,DX
4,TCGA-94-7033-01Z-00-DX1.43146ed9-30a5-420d-bd9...,23f1ad0c-c9d5-408f-bba8-1bb71364007b.rna_seq.a...,TCGA-94-7033,TCGA-94-7033-01Z,TCGA-94-7033-01A,Primary,DX
...,...,...,...,...,...,...,...
515,TCGA-66-2785-01Z-00-DX1.b9439ee1-d22b-4ccd-b53...,b5356a8b-9401-442f-a5dd-7e9273723a67.rna_seq.a...,TCGA-66-2785,TCGA-66-2785-01Z,TCGA-66-2785-01A,Primary,DX
516,TCGA-77-6843-01Z-00-DX1.5ced4995-81a1-4dfd-82b...,0b646082-9e64-4fb2-a9c9-e283d4c49852.rna_seq.a...,TCGA-77-6843,TCGA-77-6843-01Z,TCGA-77-6843-01A,Primary,DX
517,TCGA-66-2781-01Z-00-DX1.ed9ff5b7-7c66-4bf7-bfb...,59fd6a72-d96d-42af-8013-11f43af6eed5.rna_seq.a...,TCGA-66-2781,TCGA-66-2781-01Z,TCGA-66-2781-01A,Primary,DX
518,TCGA-39-5028-01Z-00-DX1.7994ec22-746d-4c30-813...,77b0bb2f-6e9f-4c67-b64c-98eebf0bd99d.rna_seq.a...,TCGA-39-5028,TCGA-39-5028-01Z,TCGA-39-5028-01A,Primary,DX


In [10]:
is_magnif = metadata.apply(lambda x: has_magnif(glob.glob(f"data/*/*/{x.image_path}")[0]), axis=1)
(~is_magnif).sum()

0

In [11]:
metadata = metadata[is_magnif]
metadata

,image_path,rna_path,case_id,sample_slide_id,sample_rna_id,sample_type,data_type_info
0,TCGA-60-2712-01Z-00-DX1.97003dfc-4b37-4491-86e...,80ca4c12-21e8-45d1-8820-537b99bce32d.rna_seq.a...,TCGA-60-2712,TCGA-60-2712-01Z,TCGA-60-2712-01A,Primary,DX
1,TCGA-56-7221-01Z-00-DX1.f897f1ee-2796-4183-931...,545f9937-b128-4f44-8b12-ded0fb79bf3f.rna_seq.a...,TCGA-56-7221,TCGA-56-7221-01Z,TCGA-56-7221-01A,Primary,DX
2,TCGA-21-A5DI-01Z-00-DX1.E9123261-ADE7-468C-9E9...,9b86812f-b1ee-4b6d-9691-8587f2487c4a.rna_seq.a...,TCGA-21-A5DI,TCGA-21-A5DI-01Z,TCGA-21-A5DI-01A,Primary,DX
3,TCGA-43-7657-01Z-00-DX1.d8a5d257-c5ca-4192-b6a...,6df80c92-775e-4bcf-b2c7-6cbd7e147447.rna_seq.a...,TCGA-43-7657,TCGA-43-7657-01Z,TCGA-43-7657-01A,Primary,DX
4,TCGA-94-7033-01Z-00-DX1.43146ed9-30a5-420d-bd9...,23f1ad0c-c9d5-408f-bba8-1bb71364007b.rna_seq.a...,TCGA-94-7033,TCGA-94-7033-01Z,TCGA-94-7033-01A,Primary,DX
...,...,...,...,...,...,...,...
515,TCGA-66-2785-01Z-00-DX1.b9439ee1-d22b-4ccd-b53...,b5356a8b-9401-442f-a5dd-7e9273723a67.rna_seq.a...,TCGA-66-2785,TCGA-66-2785-01Z,TCGA-66-2785-01A,Primary,DX
516,TCGA-77-6843-01Z-00-DX1.5ced4995-81a1-4dfd-82b...,0b646082-9e64-4fb2-a9c9-e283d4c49852.rna_seq.a...,TCGA-77-6843,TCGA-77-6843-01Z,TCGA-77-6843-01A,Primary,DX
517,TCGA-66-2781-01Z-00-DX1.ed9ff5b7-7c66-4bf7-bfb...,59fd6a72-d96d-42af-8013-11f43af6eed5.rna_seq.a...,TCGA-66-2781,TCGA-66-2781-01Z,TCGA-66-2781-01A,Primary,DX
518,TCGA-39-5028-01Z-00-DX1.7994ec22-746d-4c30-813...,77b0bb2f-6e9f-4c67-b64c-98eebf0bd99d.rna_seq.a...,TCGA-39-5028,TCGA-39-5028-01Z,TCGA-39-5028-01A,Primary,DX


In [12]:
metadata['id_pair'] = np.arange(len(metadata))

In [13]:
metadata.to_csv(f'data/metadata_{slide_type}.csv', index=False)